# 05 — Prophet Baseline Forecast

> **Objective:** Build a baseline temperature forecast using Facebook Prophet. Aggregate to daily series, train/test split, evaluate metrics (RMSE, MAE, MAPE), and generate 30-day predictions.

---

## 1. Setup and Load Data

Import libraries, load raw CSV, and validate required columns.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
import seaborn as sns

sns.set_theme(style="whitegrid")

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_raw_weather

reports_dir = project_root / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

# Load and validate data
df = load_raw_weather(project_root)

required = ["temperature_celsius", "last_updated"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df.dropna(subset=["last_updated", "temperature_celsius"]).copy()

print("Shape after dropna:", df.shape)
print("Time range:", df["last_updated"].min(), "->", df["last_updated"].max())
display(df[required + ["humidity", "wind_kph", "precip_mm"]].head(3) if "humidity" in df.columns else df[required].head(3))
print("Unique calendar days:", df["last_updated"].dt.normalize().nunique())

## 2. Daily Aggregation

Build Prophet input: one row per calendar day with `ds` (date) and `y` (mean temperature in °C).

In [6]:
df["date"] = df["last_updated"].dt.normalize()
prophet_df = df.groupby("date", as_index=False)["temperature_celsius"].mean()
prophet_df.columns = ["ds", "y"]

prophet_df = prophet_df.copy()
prophet_df["ds"] = pd.to_datetime(prophet_df["ds"])
prophet_df = prophet_df.sort_values("ds").reset_index(drop=True)

print("prophet_df shape:", prophet_df.shape)
print("ds min -> max:", prophet_df["ds"].min(), "->", prophet_df["ds"].max())
display(prophet_df.head())


prophet_df shape: (687, 2)
ds min -> max: 2024-05-16 00:00:00 -> 2026-04-03 00:00:00


,ds,y
0,2024-05-16,23.767196
1,2024-05-17,24.451648
2,2024-05-18,25.355610
3,2024-05-19,25.270103
4,2024-05-20,25.430769


## 3. Train/Test Split

Temporal holdout: last 30 days reserved for testing.

In [7]:
cutoff = prophet_df["ds"].max() - pd.Timedelta(days=30)
train = prophet_df[prophet_df["ds"] <= cutoff].copy()
test = prophet_df[prophet_df["ds"] > cutoff].copy()


def split_report(name, part):
    print(f"\n{name}: n={len(part)}")
    if len(part):
        print(f"  ds: {part['ds'].min()} -> {part['ds'].max()}")


split_report("TRAIN", train)
split_report("TEST", test)


TRAIN: n=657
  ds: 2024-05-16 00:00:00 -> 2026-03-04 00:00:00

TEST: n=30
  ds: 2026-03-05 00:00:00 -> 2026-04-03 00:00:00


## 4. Model Training and Evaluation

Fit Prophet with weekly/yearly seasonality, compute test metrics, and generate forecast.

In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
)
model.fit(train)

fc_test = model.predict(test[["ds"]].copy())
y_true = test["y"].to_numpy(dtype=float)
y_hat = fc_test["yhat"].to_numpy(dtype=float)

rmse = float(np.sqrt(mean_squared_error(y_true, y_hat)))
mae = float(mean_absolute_error(y_true, y_hat))
mask = y_true != 0
if mask.any():
    mape = float(np.mean(np.abs((y_true[mask] - y_hat[mask]) / y_true[mask])) * 100)
else:
    mape = float("nan")

print(f"Test RMSE: {rmse:.4f} °C")
print(f"Test MAE:  {mae:.4f} °C")
print(f"Test MAPE: {mape:.2f}%")

future = model.make_future_dataframe(periods=30, freq="D")
forecast = model.predict(future)

out_plot = reports_dir / "prophet_baseline_forecast.png"
fig = model.plot(forecast)
fig.set_size_inches(10, 6)
fig.savefig(out_plot, dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved:", out_plot)

out_comp = reports_dir / "prophet_baseline_components.png"
fig2 = model.plot_components(forecast)
fig2.set_size_inches(10, 8)
fig2.savefig(out_comp, dpi=150, bbox_inches="tight")
plt.close(fig2)
print("Saved:", out_comp)


18:25:41 - cmdstanpy - INFO - Chain [1] start processing
18:25:41 - cmdstanpy - INFO - Chain [1] done processing


Test RMSE: 0.7665 °C
Test MAE:  0.6865 °C
Test MAPE: 3.95%
Saved: c:\Users\lucas\Desktop\pma-weather-forecasting\reports\prophet_baseline_forecast.png
Saved: c:\Users\lucas\Desktop\pma-weather-forecasting\reports\prophet_baseline_components.png


## 5. Summary

**Results:**
- Test RMSE: 0.77°C | MAE: 0.69°C | MAPE: 3.95%

**Notes:**
- Metrics computed on held-out test rows only.
- Forecast figure covers history + 30-day future horizon.
- For production forecasts, refit on full `prophet_df` before predicting future dates.